To make the Docling example a complete RAG ingestion pipeline, we’ll add chunking + embeddings + vector storage after extracting PDF text, tables, and images.

For a simple working example, we can use:

* Docling — parse PDF text, tables, and figures.

* LangChain text splitter — split content into chunks.

* Sentence Transformers — generate embeddings locally.

* FAISS — store vectors and perform similarity search.

This avoids requiring a cloud API key for the embedding model.

## Complete code: PDF → extraction → chunking → embeddings → FAISS

### 1. Install dependencies

Bash

```
pip install docling langchain-text-splitters sentence-transformers faiss-cpu numpy
```

### 2. Complete Python implementation

Python

Run

```
from pathlib import Path

import faiss
import numpy as np
from docling.document_converter import DocumentConverter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer


PDF_PATH = "return_policy.pdf"
OUTPUT_DIR = Path("extracted_output")
IMAGE_DIR = OUTPUT_DIR / "images"

OUTPUT_DIR.mkdir(exist_ok=True)
IMAGE_DIR.mkdir(exist_ok=True)

# 1. Parse PDF
converter = DocumentConverter()
result = converter.convert(PDF_PATH)
doc = result.document

# 2. Extract document content
markdown = doc.export_to_markdown()
(OUTPUT_DIR / "document.md").write_text(
    markdown, encoding="utf-8"
)

# Create source records for text, tables and images
records = [
    {
        "text": markdown,
        "type": "document",
        "source": PDF_PATH
    }
]

# Extract tables separately for structured access
for i, table in enumerate(doc.tables):
    table_md = table.export_to_markdown(doc=doc)

    (OUTPUT_DIR / f"table_{i + 1}.md").write_text(
        table_md, encoding="utf-8"
    )

    records.append({
        "text": f"Table {i + 1}\n{table_md}",
        "type": "table",
        "source": PDF_PATH
    })

# Extract images from detected picture elements
for i, picture in enumerate(doc.pictures):
    image = picture.get_image(doc)

    if image is not None:
        image_path = IMAGE_DIR / f"image_{i + 1}.png"
        image.save(image_path)

        print(f"Image saved: {image_path}")

        # Add a caption generated by a vision model here.
        # A text embedding model cannot directly understand
        # the image pixels.
        caption = f"Figure {i + 1} from {PDF_PATH}"

        records.append({
            "text": caption,
            "type": "image_caption",
            "source": PDF_PATH,
            "image_path": str(image_path)
        })

# 3. Chunk the extracted content
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)

chunks = []

for record in records:
    if not record["text"].strip():
        continue

    parts = splitter.split_text(record["text"])

    for part in parts:
        chunks.append({
            "text": part,
            "type": record["type"],
            "source": record["source"],
            "image_path": record.get("image_path")
        })

print(f"Total chunks: {len(chunks)}")

# 4. Generate embeddings
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# 5. Store vectors in FAISS
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

# Save vector index and chunk metadata
faiss.write_index(index, "return_policy.faiss")
np.save("return_policy_chunks.npy",
        np.array(chunks, dtype=object),
        allow_pickle=True)

print(f"Indexed {index.ntotal} chunks successfully.")

# 6. Search the indexed chunks
def search(query: str, top_k: int = 3):
    query_vector = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_vector,
        min(top_k, index.ntotal)
    )

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        results.append({
            **chunks[int(idx)],
            "score": float(score)
        })

    return results


if __name__ == "__main__":
    for item in search("What is the return period?"):
        print("\nType:", item["type"])
        print("Score:", round(item["score"], 3))
        print("Content:", item["text"])
```

### 3. Understand the flow

PDF: text, tables, figures

Docling parsing and image extraction

Text normalization and chunking

Sentence Transformer embeddings

FAISS vector index

Query → top-k retrieval → LLM answer

### 4. Important production considerations

* Tables: The example indexes both the full Markdown and separate table records, so some content may be duplicated. For production, avoid duplicate indexing or use a table-aware chunking strategy.

* Images: The example extracts images but currently embeds only a placeholder caption. Call a vision model such as Gemini to generate meaningful descriptions before embedding them.

* Metadata: Preserve the actual PDF page number, section, document version, and access permissions for traceability.

* Storage: FAISS is useful for a local prototype. For a distributed Google Cloud application, evaluate Vertex AI Vector Search.

* Security: Avoid loading untrusted `.npy` files with pickle enabled; use a safer metadata format such as JSON or Parquet in production.

Interview one-liner: “I use Docling for PDF parsing, a text splitter for context-aware chunks, Sentence Transformers for embeddings, and FAISS for similarity search; image captions are generated using a vision model and indexed alongside the document metadata.”
